# Les décorateurs

In [36]:
def takeaway(meal):
    def wrapper():
        print("UpperBox")
        meal()
        print("LowerBox")
        
    return wrapper


def sandwich(garniture):
    def wrapper(*args, **kargs):
        print("/TTTTTTTT\\")
        price = garniture(*args, **kargs)
        print("\\________/")
        return price + 2
        
    return wrapper

@sandwich
def parisien():
    print("  Jambon ")
    print("  Beurre ")
    return 10

@takeaway
@sandwich
def lyonnais():
    print("  rosette ")
    print("  Beurre ")

@sandwich
def kebab(salade_tomate_oignons= True):
    if salade_tomate_oignons:
        print("Salade")
        print("Tomates")
        print("Oignons")
    print("Viande")
    return 5

In [40]:
kebab(True)

/TTTTTTTT\
Salade
Tomates
Oignons
Viande
\________/


7

In [34]:
print(parisien())

/TTTTTTTT\
  Jambon 
  Beurre 
\________/
12


In [16]:
lyonnais()

UpperBox
/TTTTTTTT\
  rosette 
  Beurre 
\________/
LowerBox


## Exercices
### exercice 1

In [44]:
import time

def time_it(func):
    def wrapper(*args, **kargs):
        start = time.time()

        value = func(*args, **kargs)
    
        end = time.time()
    
        print(f"{end - start} secondes se sont écoulées")
        return value

    return wrapper


@time_it
def func(loops=1_000_000):
    for x in range(loops):
        y = x ** 2
        
    return y

In [45]:
func()

0.05258297920227051 secondes se sont écoulées


999998000001

### Exercice 2

In [48]:
def count_calls(func):
    count = 0

    def inner(*args, **kwargs):
        result = func(*args, **kwargs)
        nonlocal count
        count += 1

        return result

    def inner_get_count():
        return count

    inner.nbcalls = inner_get_count

    return inner

In [51]:
@count_calls
def some_func():
    print("func called")

print(some_func.nbcalls())
some_func()
some_func()
print(some_func.nbcalls())

0
func called
func called
2


### Exercice 3

In [52]:
import time
from collections import OrderedDict

def cached(func):
    
    cached_values = OrderedDict()
    
    def inner_long_call(*args):

        if args in cached_values:
            result = cached_values[args]
            cached_values.move_to_end(args)

        else:

            result = func(*args)
            
            cached_values[args] = result

            if len(cached_values) > 3:
                cached_values.popitem(last=False)
        
        return result
        
    return inner_long_call

In [54]:
@time_it
@cached
def long_call(value:int):
    time.sleep(2)
    return value**2

print(long_call(54))
print(long_call(13))
print(long_call(42))
print(long_call(13))
print(long_call(54))
print(long_call(6))
print(long_call(42))

2.0059194564819336 secondes se sont écoulées
2916
2.0043129920959473 secondes se sont écoulées
169
2.0040042400360107 secondes se sont écoulées
1764
1.1205673217773438e-05 secondes se sont écoulées
169
4.0531158447265625e-06 secondes se sont écoulées
2916
2.0034430027008057 secondes se sont écoulées
36
2.0051326751708984 secondes se sont écoulées
1764


### Exercice 4
Pattern registery

In [55]:
notifications = []

def register(func):
    notifications.append(func)
    return func

@register
def notify_mail():
    print("notification on mail")

def notify_sms():
    print("notification on message")

@register
def notify_push():
    print("notification on push service")

def send_notifications():
    for notification in notifications:
        notification()

send_notifications()

notification on mail
notification on push service


## Décorateurs paramétrés

In [85]:
def deco(_func=None, *, p1=None):
    def inner_deco(func):
        def inner(*args):
            print("before func")
            print(p1)
            func(*args)
            print("after func")

        return inner

    if _func is None:
        return inner_deco
    else:
        return inner_deco(_func)
        


In [88]:
@deco
def my_func(param):
    print("in func", param)

In [89]:
my_func("toto")

before func
None
in func toto
after func


### Exercice notifications paramétrées

In [93]:
notifications = []

def register(*, level=1):
    def register_deco(func):
        notifications.append((level, func))
        return func
    return register_deco

@register(level=2)
def notify_mail():
    print("notification on mail")

@register(level=1)
def notify_sms():
    print("notification on message")

@register(level=2)
def notify_push():
    print("notification on push service")

def send_notifications(level):
    for level_notif, notification in notifications:
        if level_notif >= level:
            notification()

send_notifications(2)

notification on mail
notification on push service
